# DS2002 · Kaggle and Colab Workflow with Notebook Hygiene

**Studio — 2026-09-02 · Fall 2026**  
**Class time:** 45 minutes

---

## The bug that is not in your code

A notebook gives you one thing a script does not: the ability to run cells in any order, as many times as you like. That is why notebooks are good for exploring, and it is also the source of the single most common wrong answer in this course.

Your notebook does not remember which cells you clicked. It only remembers the values that are currently in memory. If those values came from clicking a cell twice, or from a cell you have since deleted, the notebook will happily show you a number that nobody can reproduce — including you, tomorrow.

Today we build the habits that make that impossible. Everything you write today, you will reuse in every lab and both projects.

### Watch it happen

Run the next cell once. Then run it again. Then a third time.

In [2]:
import pandas as pd

orders = pd.DataFrame({
    'game': ['NC State', 'NC State', 'NC State', 'Louisville', 'Louisville'],
    'units': [10, 10, 20, 30, 30],
})
print('starting total:', orders['units'].sum())

starting total: 100


In [3]:
# The 10% service fee. Run this cell more than once and watch.
orders['units'] = orders['units'] * 1.10
print('total now:', round(orders['units'].sum(), 2))

total now: 110.0


Each click compounds. Nothing errors, nothing looks broken, and the number is wrong in a way that survives every check except reproducing it.

The fix is not "be careful." The fix is to write cells that give the same result no matter how many times they run. That property has a name — **idempotent** — and it comes from one rule: *never overwrite your source data; derive from it.*

### Build 1 — make it idempotent

**TODO:** rewrite the fee calculation so running it five times gives the same answer as running it once. Keep `raw` untouched and compute a new column from it.

In [4]:
raw = pd.DataFrame({
    'game': ['NC State', 'NC State', 'NC State', 'Louisville', 'Louisville'],
    'units': [10, 10, 20, 30, 30],
})
# TODO: build `work` from `raw` and add a column with the fee applied,
#       without changing raw['units']
work = raw.copy()
print('starting total', work['units'].sum())
work['units'] =  work['units'] * 1.10
print('total now:', round(work['units'].sum(), 2))
# TODO: print the total of your new column

starting total 100
total now: 110.0


In [6]:
# Proof: run this and the totals must match
for attempt in range(3):
    work = raw.copy()
work['units'] =  work['units'] * 1.10
print('total now:', round(work['units'].sum(), 2))
    # TODO: apply your fee line here too, then print the total

total now: 110.0


The pattern to carry forward: one cell near the top loads the raw data and never touches it again. Every transformation produces a new frame or a new column. When a number looks wrong, you can rebuild it from the top instead of guessing at click history.

### Restart and Run All is the only real proof

Look at the numbers in the brackets to the left of your cells. Those are execution counters, and they tell the truth about what order things ran in. If `In [14]` sits above `In [3]`, the notebook you are looking at cannot be reproduced by reading it top to bottom.

Before every submission in this course: **Restart the kernel, then Run All.** If it errors, you just found a bug the grader would have found. If it changes an answer, you just found a worse one.

### Build 2 — paths that survive both environments

Kaggle and Colab put files in different places, and neither is where they sit in the repository. Hardcode one path and your notebook breaks for whoever opens it next.

Run this, then extend it.

In [7]:
import os, sys
#os checks folders
#sys checks Python environment
def data_dir():
    """First folder that actually exists, checked in order of likelihood."""
    candidates = [
        'data',                                # running inside the repo
        '../data',
        '/kaggle/input',                       # Kaggle attached dataset
        '/content/drive/MyDrive/ds2002/data',  # Colab with Drive mounted
        '/content',                            # Colab manual upload
    ]
    for path in candidates:
        if os.path.isdir(path):
            return path
    return '.'

print('environment:', 'Kaggle' if os.path.exists('/kaggle')
      else 'Colab' if 'google.colab' in sys.modules else 'other')
print('data_dir():', data_dir())

environment: Kaggle
data_dir(): /kaggle/input


**TODO:** finish `load_csv` so a missing file does not kill the notebook. Every lab in this course uses this shape: try the real file, fall back to a small inline sample, and say clearly which one you got.

In [8]:
from io import StringIO

SAMPLE = '''game,units\nNC State,1450\nLouisville,1610\n'''

def load_csv(filename, sample=SAMPLE):
    path = os.path.join(data_dir(), filename)
    for each in SAMPLE:
      if os.each.isidr(each):
        return each
    return '.'
    # TODO: if the file exists, read it and print that you used the real file.
    #       otherwise read the sample string and print that you fell back.


df = load_csv('gameday_orders.csv')
df
print('')

AttributeError: module 'os' has no attribute 'each'

### Build 3 — same input, same output

Two more things quietly break reproducibility: unseeded randomness and unrecorded versions. Run this twice and compare.

In [ ]:
import numpy as np

print('pandas', pd.__version__, '| numpy', np.__version__,
      '| python', sys.version.split()[0])

seeded = np.random.default_rng(2002).integers(0, 100, 5)
unseeded = np.random.default_rng().integers(0, 100, 5)
print('seeded:  ', seeded)
print('unseeded:', unseeded)

**TODO:** in the cell below, generate 500 random prices two ways — once seeded, once not — and print the mean of each rounded to two decimals. Run the cell twice. Which mean moves, and why does that matter when you report a number in a project?

In [ ]:
# TODO

### Keys and secrets

Nothing in this course needs an API key — Open-Meteo is open. So if a key ever appears in one of your notebooks, something has gone wrong.

When you do need one after this course:

- **Kaggle:** Add-ons -> Secrets. **Colab:** the key icon in the sidebar, then `from google.colab import userdata`.
- Read it from the environment so the value never appears in the file:

```python
import os
API_KEY = os.environ.get('SOME_API_KEY')
```

- If you do commit a key: **rotate it**. Deleting the line does not help — the old commit still has it, and anyone who cloned the repo has it too.

### Output hygiene

Your outputs get committed with the notebook, so they are part of what you hand in.

- `df.head()`, `df.shape`, `df.info()` — not `print(df)` on 40,000 rows.
- A cell that dumps ten thousand lines makes the file huge and the diff unreadable.
- Charts should be small and labeled, not a wall of default plots.

If a notebook is several megabytes, the outputs are usually why.

### The submit loop

Every graded item in this course goes out the same way:

1. Restart the kernel and Run All. Confirm zero errors.
2. Check the outputs you are leaving in are the ones you want read.
3. Save, then get the file into your repo in the right folder with the right name.
4. Commit with a message that says what the work was, and push.
5. In Canvas, submit the repository link and the notebook link.

**TODO:** finish `submission_check` so it flags anything still unfilled. You will paste this into later labs, so make it strict.

In [ ]:
def submission_check(items):
    """items: dict of label -> the value you filled in.
    A value fails if it is None, empty, or still says TODO."""
    all_ok = True
    for label, value in items.items():
        # TODO: set `ok` False when value is None, an empty string,
        #       or a string that starts with 'TODO'
        ok = True
        print('PASS' if ok else 'FAIL', '-', label)
        all_ok = all_ok and ok
    print()
    print('Ready to submit:', all_ok)
    return all_ok

submission_check({
    'repo URL': 'TODO: paste it',
    'ran restart + run all': True,
    'notebook name follows convention': '2026-09-02 — ... — Studio.ipynb',
})

---

## Checkpoint (participation)

Report the total your idempotent cell produced after three runs, and the folder `data_dir()` resolved to in your environment.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint — run your Build 1 proof cell three times first
total_after_three_runs = None      # TODO: the number, same all three times
resolved_data_dir = 'TODO'         # TODO: what data_dir() printed for you
habit_i_am_changing = 'TODO'       # TODO: one thing you will do differently

print('total after 3 runs:', total_after_three_runs)
print('data_dir():', resolved_data_dir)
print('changing:', habit_i_am_changing)